In [20]:
import pandas as pd

# read data
df = pd.read_csv("../data/dirty/ILO/EAR_4MTH_SEX_CUR_NB_A-20250408T0015.csv")

# regex
df["sex"] = df["sex.label"].str.extract(r"Sex: (\w)")[0]
df["ISO"] = df["note_indicator.label"].str.extract(r"Currency: (\w{3})")[0]
df["currency"] = df["classif1.label"].str.extract(r"Currency: (.+)")[0]
df["local_currency"] = (
    df["note_indicator.label"].str.extract(r"Currency: \w{3} - ([^|]+)")[0].str.strip()
)
df["break"] = df["note_indicator.label"].str.contains("Break in series", na=False)

# remove irrelevant data
df.drop(
    columns=[
        "obs_status.label",
        "note_classif.label",
        "note_source.label",
        "indicator.label",
        "note_indicator.label",
        "source.label",
        "sex.label",
        "classif1.label",
    ],
    inplace=True,
)

# rename columns
df.rename(
    columns={
        "ref_area.label": "country",
        "time": "year",
        "obs_value": "wage",
        "ISO": "ISO",
    },
    inplace=True,
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")

# index max year per country and sex
idx = df.groupby(["country", "sex", "currency"])["year"].idxmax()
# df = df.loc[idx].reset_index(drop=True)

# last_break = df[df["break"]].groupby("ISO")["year"].max().rename("last_break")
# df = df.merge(last_break, on="ISO", how="left")
# df["last_break"] = pd.to_numeric(df["last_break"], errors="coerce")

map = {"U.S. dollars": "USD", "2021 PPP $": "PPP", "Local currency": "Local"}
df["currency"] = df["currency"].replace(map)

df.to_csv("../data/clean/ILO.csv")

In [28]:
df_ratio = df.pivot_table(
    index=["ISO", "year", "country", "currency", "local_currency"],
    columns="sex",
    values="wage",
    aggfunc="first",
).reset_index()

break_info = df[
    [
        "ISO",
        "year",
        "break",
        # "last_break",
    ]
].drop_duplicates(subset=["ISO", "year"])

df_ratio = df_ratio.merge(break_info, on=["ISO", "year"], how="left")

df_ratio["ratio"] = df_ratio["F"] / df_ratio["M"]

df_ratio = df_ratio[
    [
        "ISO",
        "year",
        "country",
        "T",
        "M",
        "F",
        "ratio",
        "break",
        "local_currency",
        # "last_break",
        "currency",
    ]
]
df_ratio

,ISO,year,country,T,M,F,ratio,break,local_currency,currency
0,ABW,2010,Aruba,3013.000,3338.000,2713.000,0.812762,False,Florin (AWG),Local
1,ABW,2010,Aruba,1860.382,2061.054,1675.147,0.812762,False,Florin (AWG),PPP
2,ABW,2010,Aruba,1683.240,1864.804,1515.642,0.812762,False,Florin (AWG),USD
3,AFG,2014,Afghanistan,8848.569,9135.149,5517.315,0.603966,False,Afghani (AFN),Local
4,AFG,2014,Afghanistan,517.709,534.476,322.805,0.603965,False,Afghani (AFN),PPP
...,...,...,...,...,...,...,...,...,...,...
8411,ZWE,2022,Zimbabwe,212.044,223.145,193.270,0.866118,False,Zimbabwean dollar (ZWL),PPP
8412,ZWE,2022,Zimbabwe,80.864,85.097,73.704,0.866117,False,Zimbabwean dollar (ZWL),USD
8413,ZWE,2023,Zimbabwe,406978.727,420431.180,385130.787,0.916038,False,Zimbabwean dollar (ZWL),Local
8414,ZWE,2023,Zimbabwe,1711.902,1768.488,1620.001,0.916037,False,Zimbabwean dollar (ZWL),PPP


In [ ]:
df_wide = df_ratio.pivot_table(
    index=["ISO", "year", "country", "break"],
    columns="currency",
    values=["F", "M", "T", "ratio"],
    aggfunc="first",
).reset_index()

df_wide.columns = [
    f"{metric}_{curr.lower()}" if curr else metric for metric, curr in df_wide.columns
]

print(df_wide)
df_wide.to_csv("../data/clean/ILO_ratio_wide.csv")

      ISO  year      country  break     F_local     F_ppp     F_usd  \
0     ABW  2010        Aruba  False    2713.000  1675.147  1515.642   
1     AFG  2014  Afghanistan  False    5517.315   322.805    96.376   
2     AFG  2020  Afghanistan   True   10843.294   686.841   141.164   
3     AGO  2019       Angola   True   58793.485   404.360   161.155   
4     AGO  2021       Angola  False   69497.802   341.196   110.062   
...   ...   ...          ...    ...         ...       ...       ...   
3576  ZWE  2014     Zimbabwe  False     283.740       NaN       NaN   
3577  ZWE  2015     Zimbabwe  False         NaN       NaN       NaN   
3578  ZWE  2021     Zimbabwe  False   14481.879   191.960   163.540   
3579  ZWE  2022     Zimbabwe  False   27635.835   193.270    73.704   
3580  ZWE  2023     Zimbabwe  False  385130.787  1620.001   109.750   

         M_local     M_ppp     M_usd     T_local     T_ppp     T_usd  \
0       3338.000  2061.054  1864.804    3013.000  1860.382  1683.240   
1  

In [23]:
df_ratio_local = pd.DataFrame(
    df_ratio[
        (df_ratio["currency"] == "Local")
        & df_ratio["F"].notna()
        & df_ratio["M"].notna()
    ]
)

df_ratio_local["female_share"] = (df_ratio_local["T"] - df_ratio_local["M"]) / (
    df_ratio_local["F"] - df_ratio_local["M"]
)
df_ratio_local["female_share"] = df_ratio_local["female_share"].clip(0, 1)
df_ratio_local["male_share"] = 1 - df_ratio_local["female_share"]

df_ratio_local["balanced_avg"] = (df_ratio_local["F"] + df_ratio_local["M"]) / 2
df_ratio_local["balanced_ratio"] = df_ratio_local["F"] / df_ratio_local["balanced_avg"]

df_ratio_local.drop(columns="currency", inplace=True)
df_ratio_local.to_csv("../data/clean/ILO_ratio_local.csv")

In [24]:
low_q, high_q = df_ratio["ratio"].quantile([0.1, 0.9])

# 2) Filter out the top and bottom 10%
df_mid10 = df_ratio.loc[df_ratio["ratio"].between(low_q, high_q),].reset_index(
    drop=True
)

df_mid10.to_csv("../data/clean/ILO_ratio.csv")

In [ ]:
from currency_converter import CurrencyConverter
from datetime import date

# build your converter once
c = CurrencyConverter()

# assume your raw df has:
#  - currency == 'Local' or 'USD'
#  - wage      (the avg monthly earnings)
#  - local_currency_code  (e.g. 'AFN','USD','EUR', etc.)
#  - year     (numeric)
#  - ISO, country, break, last_break, sex

# 1) take only the Local rows that lack USD equivalents
df_local = df[df["currency"] == "Local"].copy()

# 2) convert each local wage into USD using the rate on Jan 1 of that year
def to_usd(row):
    try:
        rate = c.convert(
            1,
            row["local_currency_code"],
            "USD",
            date=int(row["year"]), 1, 1
        )
        return row["wage"] * rate
    except Exception:
        return None
